# 01 · Reconstrucción de traza — IVR Alkosto

En el pipeline original, todos los pasos de una interacción (navegación,
validaciones de flujo, llamadas a webservice, paso a asesor) llegaban mezclados
en una sola columna (`opcionesnavegaciontrazaopciones`). En los datos actuales,
la vista los separa: `traza_opciones` trae la navegación principal y cada
columna `customN` trae **un solo paso** adicional (validaciones, resultados de
webservice, etc.).

Confirmamos contra el PDF del flujo (`Flujo_actualizado_Alkosto_corte_28-04-26`)
que esos pasos de `customN` **sí son nodos de decisión reales** del árbol (carriles
"API": `¿Error de consulta?`, `¿Estado Fraude?`, `¿Tiene PEA?`, etc.), así que se
integran todos — no se excluyen.

**Entrada:** parquet generado por `00_extraccion.ipynb`.

**Salida:**
- `df_pasos_<periodo>.parquet` — formato largo (un paso por fila), ya con el orden
  cronológico correcto. Este es el insumo directo del Notebook 3 (reemplaza el
  `split('|')` + `stack()` que hacía `01_Alk_final.ipynb` sobre la columna cruda).
- `df_traza_completa_<periodo>.parquet` — un renglón por `id_conversacion` con la
  traza reconstruida como string `|codigo;texto;tiempo|...`, solo para auditoría /
  comparación visual contra `traza_opciones` original.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
tqdm.pandas()

## Parámetros — deben coincidir con los usados en `00_extraccion.ipynb`

In [ ]:
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
RAW_DIR = DATA_DIR / "00_raw"
STAGE_DIR = DATA_DIR / "01_staging"
STAGE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / f"df_raw_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_PASOS_PATH = STAGE_DIR / f"df_pasos_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_TRAZA_PATH = STAGE_DIR / f"df_traza_completa_{FECHA_INICIO}_{FECHA_FIN}.parquet"

assert INPUT_PATH.exists(), f"No encuentro {INPUT_PATH} — corre primero 01_extraccion.ipynb"
print(f"Leyendo: {INPUT_PATH}")

Leyendo: data\00_raw\df_raw_2026-06-01_2026-06-30.parquet


In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(df.shape)

ID_COL = "id_conversacion"
COL_TRAZA_PRINCIPAL = "traza_opciones"
columnas_custom = sorted(
    [c for c in df.columns if c.startswith("custom") and c not in ("custom_50",)],
    key=lambda c: int(c.replace("custom", "")) if c.replace("custom", "").isdigit() else 999,
)
print("Columnas fuente que se van a integrar:", [COL_TRAZA_PRINCIPAL] + columnas_custom)

(105499, 42)
Columnas fuente que se van a integrar: ['traza_opciones', 'custom2', 'custom3', 'custom11', 'custom15', 'custom16', 'custom21', 'custom22', 'custom24', 'custom25', 'custom26', 'custom28', 'custom29', 'custom30', 'custom31', 'custom32', 'custom33', 'custom34', 'custom47', 'custom48']


## Parseo de un campo tipo `|codigo;texto;tiempo|codigo;texto;tiempo...`

Tanto `traza_opciones` (varios pasos) como cada `customN` (un solo paso) usan el
mismo formato interno, así que se parsean con la misma función.

In [ ]:
def parsear_pasos(valor):
    """Convierte '|cod;texto;tiempo|cod;texto;tiempo' en una lista de tuplas
    (codigo, texto, tiempo). Ignora vacíos y fragmentos mal formados (no 3 partes).
    Devuelve también el conteo de fragmentos descartados por mal formados.
    """
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return [], 0
    partes = str(valor).split("|")
    pasos = []
    descartados = 0
    for parte in partes:
        parte = parte.strip()
        if not parte:
            continue
        campos = parte.split(";")
        if len(campos) != 3:
            descartados += 1
            continue
        codigo, texto, tiempo = (c.strip() for c in campos)
        pasos.append((codigo, texto, tiempo))
    return pasos, descartados

## Reconstrucción por `id_conversacion`

Por cada fila: se juntan los pasos de `traza_opciones` + todas las `customN` no
nulas, etiquetando de dónde vino cada paso (`origen_campo`), y se ordenan por el
código numérico inicial para recuperar la secuencia cronológica real.

In [ ]:
registros_pasos = []  # una fila por paso -> formato largo
registros_traza = []  # una fila por id_conversacion -> string reconstruido, para auditoría
total_descartados = 0
total_codigo_no_numerico = 0

columnas_fuente = [COL_TRAZA_PRINCIPAL] + columnas_custom

for row in tqdm(df[[ID_COL] + columnas_fuente].itertuples(index=False), total=len(df)):
    id_conv = getattr(row, ID_COL)
    pasos_fila = []  # (codigo_str, codigo_num, texto, tiempo, origen_campo)

    for campo in columnas_fuente:
        valor = getattr(row, campo)
        pasos, descartados = parsear_pasos(valor)
        total_descartados += descartados
        for codigo, texto, tiempo in pasos:
            codigo_num = pd.to_numeric(codigo, errors="coerce")
            if pd.isna(codigo_num):
                total_codigo_no_numerico += 1
            pasos_fila.append((codigo, codigo_num, texto, tiempo, campo))

    if not pasos_fila:
        continue

    # Orden cronológico: por código numérico; los no-numéricos van al final,
    # conservando el orden relativo en que llegaron (sort estable).
    pasos_fila.sort(key=lambda p: (pd.isna(p[1]), p[1] if not pd.isna(p[1]) else 0))

    for orden, (codigo, codigo_num, texto, tiempo, origen_campo) in enumerate(pasos_fila):
        registros_pasos.append(
            {
                "id_conversacion": id_conv,
                "orden": orden,
                "op_num": codigo,
                "op_text": texto,
                "op_tiempo": tiempo,
                "origen_campo": origen_campo,
            }
        )

    traza_reconstruida = "".join(f"|{c};{t};{tp}" for c, _, t, tp, _ in pasos_fila)
    registros_traza.append(
        {
            "id_conversacion": id_conv,
            "traza_completa": traza_reconstruida,
            "n_pasos": len(pasos_fila),
            "n_pasos_custom": sum(1 for p in pasos_fila if p[4] != COL_TRAZA_PRINCIPAL),
        }
    )

df_pasos = pd.DataFrame(registros_pasos)
df_traza_completa = pd.DataFrame(registros_traza)

print(f"Fragmentos mal formados descartados: {total_descartados}")
print(f"Pasos con código no numérico (van al final del orden): {total_codigo_no_numerico}")
print(f"id_conversacion sin ningún paso parseable: {len(df) - len(registros_traza)}")

100%|██████████| 105499/105499 [00:49<00:00, 2149.89it/s]


Fragmentos mal formados descartados: 0
Pasos con código no numérico (van al final del orden): 0
id_conversacion sin ningún paso parseable: 8187


## Chequeos de sanidad

Antes de pasar esto al Notebook 3, valida que la integración de `customN` sí esté
aportando pasos (si `n_pasos_custom` fuera 0 en todas las filas, algo falló en el
aplanado del Notebook 1) y que el orden por código no esté generando secuencias
absurdas.

In [ ]:
print("Filas en df_pasos (formato largo):", len(df_pasos))
print("id_conversacion cubiertos:", df_pasos["id_conversacion"].nunique(), "/", df[ID_COL].nunique())
print("\nDistribución de origen_campo:")
print(df_pasos["origen_campo"].value_counts())

print("\n% de conversaciones con al menos un paso proveniente de customN:")
print((df_traza_completa["n_pasos_custom"] > 0).mean().round(3))

print("\nDistribución de cantidad de pasos por conversación:")
print(df_traza_completa["n_pasos"].describe())

Filas en df_pasos (formato largo): 1296067
id_conversacion cubiertos: 97312 / 105499

Distribución de origen_campo:
origen_campo
traza_opciones    818010
custom29           82069
custom16           82010
custom28           46034
custom2            45699
custom34           39896
custom11           35366
custom21           31358
custom24           27810
custom3            27411
custom30           15783
custom25           12139
custom15           11694
custom22            8803
custom32            5166
custom47            3638
custom48            1383
custom31             817
custom33             817
custom26             164
Name: count, dtype: int64

% de conversaciones con al menos un paso proveniente de customN:
0.942

Distribución de cantidad de pasos por conversación:
count    97312.000000
mean        13.318676
std          7.501728
min          1.000000
25%          8.000000
50%         12.000000
75%         17.000000
max         60.000000
Name: n_pasos, dtype: float64


In [ ]:
# Inspección manual de un caso con pasos custom, para comparar visualmente
# contra la traza_opciones original y confirmar que el orden quedó coherente.
ejemplo_id = df_traza_completa.loc[df_traza_completa["n_pasos_custom"] > 0, "id_conversacion"].iloc[0]
print("Ejemplo:", ejemplo_id)
print("\ntraza_opciones original:")
print(df.loc[df[ID_COL] == ejemplo_id, COL_TRAZA_PRINCIPAL].values[0])
print("\nTraza reconstruida (orden final):")
df_pasos[df_pasos["id_conversacion"] == ejemplo_id].sort_values("orden")

Ejemplo: 891f883b-e8c0-4a43-91a8-00f93459a745

traza_opciones original:
|0;Inicio IVR ;0|3;Habeas data positivo;48030|17;Menu principal;346|7;Garantias y devoluciones ;24254|14;Repetir informacion;26061|997;Repeat;23|523;Iniciar_Tu_Garantia;9781|527;Igual_O_Menor_30_Dias;8972|540;Producto_Deteriorado;5824|528;Gran_Tamano;14839|204;Paso agente garantias;132|900;Bienvenida encuesta SAC;569944|901;Primera pregunta SAC;38|902;Segunda pregunta SAC;16857|906;Pasa a buzon = NO;13125|1003;Finalización por fin de flujo;31

Traza reconstruida (orden final):


,id_conversacion,orden,op_num,op_text,op_tiempo,origen_campo
0,891f883b-e8c0-4a43-91a8-00f93459a745,0,0,Inicio IVR,0,traza_opciones
1,891f883b-e8c0-4a43-91a8-00f93459a745,1,2,Numero documento ingresado,1023941473,custom2
2,891f883b-e8c0-4a43-91a8-00f93459a745,2,3,Habeas data positivo,48030,traza_opciones
3,891f883b-e8c0-4a43-91a8-00f93459a745,3,7,Garantias y devoluciones,24254,traza_opciones
4,891f883b-e8c0-4a43-91a8-00f93459a745,4,12,Usuario_Identificado_Con_ANI,NO,custom16
5,891f883b-e8c0-4a43-91a8-00f93459a745,5,14,Repetir informacion,26061,traza_opciones
6,891f883b-e8c0-4a43-91a8-00f93459a745,6,17,Menu principal,346,traza_opciones
7,891f883b-e8c0-4a43-91a8-00f93459a745,7,100,Consulta ws ConsultaHabeasData,FAILURE,custom22
8,891f883b-e8c0-4a43-91a8-00f93459a745,8,101,Consulta ws ActualizaHabeasData,FAILURE,custom21
9,891f883b-e8c0-4a43-91a8-00f93459a745,9,107,Consulta ws CheckAftersalesCases,OK,custom28


Revisa el ejemplo impreso arriba: los pasos que vinieron de columnas `customN`
deben aparecer intercalados en la posición cronológica correcta (por código), no
todos al final. Si ves algo raro (por ejemplo, un paso de validación apareciendo
antes de `Inicio IVR`), revisa `total_codigo_no_numerico` — puede haber códigos
con formato distinto que necesiten un tratamiento especial antes de ordenar.

## Exportar

In [ ]:
df_pasos.to_parquet(OUT_PASOS_PATH, index=False)
df_traza_completa.to_parquet(OUT_TRAZA_PATH, index=False)
print(f"Guardado: {OUT_PASOS_PATH}  ({len(df_pasos)} filas)")
print(f"Guardado: {OUT_TRAZA_PATH}  ({len(df_traza_completa)} filas)")

# Muestra Excel para revisión manual
muestra_path = STAGE_DIR / f"df_pasos_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_pasos.head(500).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

Guardado: data\01_staging\df_pasos_2026-06-01_2026-06-30.parquet  (1296067 filas)
Guardado: data\01_staging\df_traza_completa_2026-06-01_2026-06-30.parquet  (97312 filas)
Muestra Excel: data\01_staging\df_pasos_muestra_2026-06-01_2026-06-30.xlsx
